In [10]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh
display_scale = 0.75

In [8]:
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


def eye_aspect_ratio(landmarks, eye_indices, w, h):
    pts = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in eye_indices]

    p1, p2, p3, p4, p5, p6 = pts

    vertical_1 = euclidean(p2, p6)
    vertical_2 = euclidean(p3, p5)
    horizontal = euclidean(p1, p4)

    if horizontal == 0:
        return 0

    return (vertical_1 + vertical_2) / (2.0 * horizontal)


LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

blink_threshold = 0.21

In [ ]:
# Testing blink detection

cap = cv2.VideoCapture(0)

blink_count = 0
blink_threshold = 0.21
eye_closed = False

window_name = "Blink Detection Test"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 900, 600)

display_scale = 0.75

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark

            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f"Blinks: {blink_count}", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            if avg_ear < blink_threshold and not eye_closed:
                eye_closed = True

            if avg_ear >= blink_threshold and eye_closed:
                blink_count += 1
                eye_closed = False

        # resize only for display
        display_frame = cv2.resize(
            frame,
            None,
            fx=display_scale,
            fy=display_scale
        )

        cv2.imshow(window_name, display_frame)
        # press 'q' to quit
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        # or quit if window is closed normally
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776679950.385021 2155485 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776679950.443616 2172990 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776679950.444782 2172984 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776679950.449093 2172977 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [9]:
def get_point(landmarks, idx, w, h):
    return np.array([
        int(landmarks[idx].x * w),
        int(landmarks[idx].y * h)
    ])

turn_threshold = 0.075

In [ ]:
# Testing head turn detection

cap = cv2.VideoCapture(0)

window_name = "Head Turn Detection Test"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 900, 600)

display_scale = 0.75

turn_threshold = 0.075

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        direction = "No face"

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark

            nose = get_point(face_landmarks, 1, w, h)
            left_face = get_point(face_landmarks, 234, w, h)
            right_face = get_point(face_landmarks, 454, w, h)

            face_center_x = (left_face[0] + right_face[0]) / 2
            nose_offset = (nose[0] - face_center_x) / w

            if nose_offset < -turn_threshold:
                direction = "Turned Left"
            elif nose_offset > turn_threshold:
                direction = "Turned Right"
            else:
                direction = "Forward"

            # draw landmarks used
            cv2.circle(frame, tuple(nose), 4, (0, 0, 255), -1)
            cv2.circle(frame, tuple(left_face), 4, (255, 0, 0), -1)
            cv2.circle(frame, tuple(right_face), 4, (255, 0, 0), -1)

            cv2.putText(frame, f"Offset: {nose_offset:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.putText(frame, f"Direction: {direction}", (30, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        display_frame = cv2.resize(frame, None, fx=display_scale, fy=display_scale)
        cv2.imshow(window_name, display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776680153.005120 2155485 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776680153.067639 2175797 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776680153.069350 2175787 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776680153.076107 2175782 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [19]:
# Combination of head turn and blink detection

cap = cv2.VideoCapture(0)

window_name = "Fixed Liveness Challenge"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 900, 600)

blink_count = 0
eye_closed = False
challenge_step = 1
liveness_passed = False

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        avg_ear = 0.0
        direction = "No face"

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark


            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            # ----- Head turn detection -----
            nose = get_point(face_landmarks, 1, w, h)
            left_face = get_point(face_landmarks, 234, w, h)
            right_face = get_point(face_landmarks, 454, w, h)

            face_center_x = (left_face[0] + right_face[0]) / 2
            nose_offset = (nose[0] - face_center_x) / w

            if nose_offset < -turn_threshold:
                direction = "Turned Left"
            elif nose_offset > turn_threshold:
                direction = "Turned Right"
            else:
                direction = "Forward"

            # Draw debug points
            cv2.circle(frame, tuple(nose), 4, (0, 0, 255), -1)
            cv2.circle(frame, tuple(left_face), 4, (255, 0, 0), -1)
            cv2.circle(frame, tuple(right_face), 4, (255, 0, 0), -1)

            if challenge_step == 1:
                instruction = "Step 1: Turn head left"

                if direction == "Turned Left":
                    challenge_step = 2
                    blink_count = 0
                    eye_closed = False

            elif challenge_step == 2:
                instruction = "Step 2: Blink once"

                if avg_ear < blink_threshold and not eye_closed:
                    eye_closed = True

                if avg_ear >= blink_threshold and eye_closed:
                    blink_count += 1
                    eye_closed = False

                if blink_count >= 1:
                    challenge_step = 3
                    liveness_passed = True

            else:
                instruction = "Liveness Passed"

            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            cv2.putText(frame, f"Blinks: {blink_count}", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
            cv2.putText(frame, f"Direction: {direction}", (30, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 255), 2)
            cv2.putText(frame, instruction, (30, 170),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

            if liveness_passed:
                cv2.putText(frame, "Verified as live user", (30, 220),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)
        else:
            cv2.putText(frame, "No face detected", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

        display_frame = cv2.resize(frame, None, fx=display_scale, fy=display_scale)
        cv2.imshow(window_name, display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776684198.648047 2190326 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776684198.697650 2233233 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776684198.700339 2233222 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776684198.711286 2233225 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [23]:
# Added randomized challenge with delays between steps
# Press 'r' to try another challenge
import random
import time

def make_random_challenge():
    blink_target = random.randint(1, 3)

    options = [
        [{"type": "turn_left"}, {"type": "blink", "count": blink_target}],
        [{"type": "turn_right"}, {"type": "blink", "count": blink_target}],
        [{"type": "turn_left"}, {"type": "forward"}, {"type": "turn_right"}],
    ]

    return random.choice(options)


def step_to_instruction(step):
    if step["type"] == "turn_left":
        return "Turn head left"
    elif step["type"] == "turn_right":
        return "Turn head right"
    elif step["type"] == "forward":
        return "Face forward"
    elif step["type"] == "blink":
        count = step["count"]
        if count == 1:
            return "Face forward and blink once"
        return f"Face forward and blink {count} times"
    return ""


def make_new_state(step_delay_seconds):
    return {
        "challenge": make_random_challenge(),
        "step": 0,
        "passed": False,
        "waiting_until": time.time() + step_delay_seconds,
        "blink_count": 0,
        "closed_frames": 0,
    }


cap = cv2.VideoCapture(0)

window_name = "Randomized Liveness Challenge"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 1000, 700)

forward_tolerance = 0.03
min_closed_frames = 2
step_delay_seconds = 1.5

state = make_new_state(step_delay_seconds)
current_instruction = None
print("Selected challenge:", [step_to_instruction(s) for s in state["challenge"]])

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        avg_ear = 0.0
        direction = "No face"
        nose_offset = 0.0

        now = time.time()

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark

            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            nose = get_point(face_landmarks, 1, w, h)
            left_face = get_point(face_landmarks, 234, w, h)
            right_face = get_point(face_landmarks, 454, w, h)

            face_center_x = (left_face[0] + right_face[0]) / 2
            nose_offset = (nose[0] - face_center_x) / w

            # swap labels here if needed
            if nose_offset < -turn_threshold:
                direction = "Turned Left"
            elif nose_offset > turn_threshold:
                direction = "Turned Right"
            elif abs(nose_offset) <= forward_tolerance:
                direction = "Forward"
            else:
                direction = "Slight Turn"

            cv2.circle(frame, tuple(nose), 4, (0, 0, 255), -1)
            cv2.circle(frame, tuple(left_face), 4, (255, 0, 0), -1)
            cv2.circle(frame, tuple(right_face), 4, (255, 0, 0), -1)

            if not state["passed"]:
                if now < state["waiting_until"]:
                    current_instruction = None
                else:
                    if state["step"] >= len(state["challenge"]):
                        state["passed"] = True
                        current_instruction = None
                    else:
                        current_instruction = state["challenge"][state["step"]]

                        if current_instruction["type"] == "turn_left":
                            if direction == "Turned Left":
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds

                        elif current_instruction["type"] == "turn_right":
                            if direction == "Turned Right":
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds

                        elif current_instruction["type"] == "forward":
                            if direction == "Forward":
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds

                        elif current_instruction["type"] == "blink":
                            if direction == "Forward":
                                if avg_ear < blink_threshold:
                                    state["closed_frames"] += 1
                                else:
                                    if state["closed_frames"] >= min_closed_frames:
                                        state["blink_count"] += 1
                                    state["closed_frames"] = 0
                            else:
                                state["closed_frames"] = 0

                            if state["blink_count"] >= current_instruction["count"]:
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds
                                state["blink_count"] = 0
                                state["closed_frames"] = 0

                    if state["step"] >= len(state["challenge"]):
                        state["passed"] = True
                        current_instruction = None
                    else:
                        if now < state["waiting_until"]:
                            current_instruction = None
                        else:
                            current_instruction = state["challenge"][state["step"]]

            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Direction: {direction}", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
            cv2.putText(frame, f"Offset: {nose_offset:.3f}", (30, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
            cv2.putText(frame, f"Step: {state['step']}/{len(state['challenge'])}", (30, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

            if current_instruction is not None and not state["passed"]:
                instruction_text = step_to_instruction(current_instruction)
                if current_instruction["type"] == "blink":
                    instruction_text += f" ({state['blink_count']}/{current_instruction['count']})"

                cv2.putText(frame, instruction_text, (30, 210),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

            if now < state["waiting_until"] and not state["passed"]:
                remaining = state["waiting_until"] - now
                cv2.putText(frame, f"Next step in {remaining:.1f}s", (30, 250),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 0), 2)

            if state["passed"]:
                cv2.putText(frame, "Liveness Passed", (30, 300),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)

        else:
            cv2.putText(frame, "No face detected - restarting challenge", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

            state = make_new_state(step_delay_seconds)
            current_instruction = None

        display_frame = cv2.resize(frame, None, fx=display_scale, fy=display_scale)
        cv2.imshow(window_name, display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("r"):
            state = make_new_state(step_delay_seconds)
            current_instruction = None
            print("New challenge:", [step_to_instruction(s) for s in state["challenge"]])

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

Selected challenge: ['Turn head left', 'Face forward and blink 2 times']


I0000 00:00:1776684569.803349 2190326 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776684569.852922 2238625 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776684569.854734 2238613 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776684569.863736 2238616 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
